In [1]:
import os
import sys
import random

In [2]:
curr_dir = os.getcwd()
par_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(par_dir)
sys.path.append(proj_dir)

In [3]:
from code_mutation.mutation_functions import CodeMutator
from database import MongoDBHelper
from code_generation.code_generation_tester import CodeGenerationHumanEvalHelper

ImportError: cannot import name 'ASTNodeTransformers' from 'code_mutation.ast_mutation' (/Users/jin/Documents/GitHub/Code Reasoning Model Research Project/code_mutation/ast_mutation.py)

In [ ]:
# %%script false --no-raise-error

db = MongoDBHelper()
base_qns_db = db.client["Base_Questions_DB"]
question_database = base_qns_db['HumanEval_Open_Ended']

In [ ]:
c = 0
f1 = 0
f2 = 0
mutate_failure = {}
f3 = 0
n_f = 0

skippers = set(["HumanEvalo72"])

for i in range(question_database.count_documents({})):
    task_id = f"HumanEvalo{i}"
    if task_id in skippers:
        continue
    qn = question_database.find_one({"_id": task_id})

    complete_sol = qn['qn'] + "\n" + qn["canon_solution"]
    check = qn['check']
    examples = qn['examples']
    
    example = random.choice(list(examples.keys()))
    func_name = CodeGenerationHumanEvalHelper.extract_func_name_from_example(example)    

    if "for" not in complete_sol:
        n_f += 1
        continue

    try:
        namespace = {}
        exec(complete_sol, namespace)
        exec(check, namespace)
        namespace['check'](namespace[func_name])

    except:
        print(f"{task_id} complete solution has issues")
        f1+=1
    
    mutator = CodeMutator()
    try: 
        mutated_code = CodeMutator.mutate_for_to_while(complete_sol)
    except Exception as e:
        print(f"{task_id}: Could not mutate the code due to the following error > {type(e),e}")
        mutate_failure[type(e)] = mutate_failure.get(type(e), 0)+1
        f2+=1
        continue    

    try:
        namespace = {}
        exec(mutated_code, namespace)
        exec(check, namespace)
        namespace['check'](namespace[func_name])
        c += 1

    except Exception as e:
        print(f"{task_id} mutated solution has issues > {e}")
        f3 += 1
    except KeyboardInterrupt as e:
        print("###",task_id)
    pass

HumanEvalo0: Could not mutate the code due to the following error > (<class 'AttributeError'>, AttributeError("type object 'ASTNodeTranformers' has no attribute 'ForToWhileNodeTransformer'"))
HumanEvalo1: Could not mutate the code due to the following error > (<class 'AttributeError'>, AttributeError("type object 'ASTNodeTranformers' has no attribute 'ForToWhileNodeTransformer'"))
HumanEvalo3: Could not mutate the code due to the following error > (<class 'AttributeError'>, AttributeError("type object 'ASTNodeTranformers' has no attribute 'ForToWhileNodeTransformer'"))
HumanEvalo4: Could not mutate the code due to the following error > (<class 'AttributeError'>, AttributeError("type object 'ASTNodeTranformers' has no attribute 'ForToWhileNodeTransformer'"))
HumanEvalo5: Could not mutate the code due to the following error > (<class 'AttributeError'>, AttributeError("type object 'ASTNodeTranformers' has no attribute 'ForToWhileNodeTransformer'"))
HumanEvalo6: Could not mutate the code d

In [ ]:
print(f"{c} test cases were mutated successfully.")
print(f"{f1} test cases failed as complete solution failed.")
print(f"{f2} test cases failed as mutator failed to mutate.")
for key in mutate_failure:
    print(f"    - {key}: {mutate_failure[key]} failures")
print(f"{f3} test cases failed as mutated code failed to pass check.")
print(f"{n_f} test cases contain no for loops for mutation.")
print(c + f1+f2+f3 + n_f +len(skippers)== question_database.count_documents({}))

0 test cases were mutated successfully.
0 test cases failed as complete solution failed.
109 test cases failed as mutator failed to mutate.
    - <class 'AttributeError'>: 109 failures
0 test cases failed as mutated code failed to pass check.
51 test cases contain no for loops for mutation.
True
